# Complexhibit — Nivel 1 (fichas de obra y exposiciones), versión 1.1

**Por qué esta versión.** En la versión 1.0, el texto se dividía en fragmentos usando la anotación manual, de modo que ningún fragmento cortaba una ficha. Esa ayuda no existe cuando se etiqueta un catálogo nuevo, por lo que la evaluación sobrestimaba el rendimiento real del nivel 1.

En la versión 1.1, el texto se divide **siempre sin mirar la anotación**, igual en el entrenamiento, en la evaluación y en el uso real. La anotación manual se sigue usando para enseñar al modelo y para evaluarlo.

Este cuaderno:
1. evalúa el modelo publicado (v1.0) en las condiciones de uso real, como referencia;
2. entrena el modelo nuevo (v1.1) con XLM-RoBERTa large y la misma partición de catálogos;
3. lo evalúa igual, por bloque exacto y por solapamiento;
4. lo publica en Hugging Face como nueva versión de `complexhibit-nivel1` solo si mejora a la 1.0;
5. guarda las predicciones de los 100 catálogos y las métricas en `resultados_nivel1_v11.zip`.

**Entrada:** ZIP con los XML de GATE. **Requisitos:** GPU L4 o A100 y token de Hugging Face **con permiso de escritura** en la organización. **Duración:** unos 40-60 minutos.

Ejecuta todo con Entorno de ejecución → Ejecutar todas.

In [ ]:
!pip -q install -U transformers datasets seqeval accelerate sentencepiece huggingface_hub
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO HAY GPU: cambia el tipo de entorno')

In [ ]:
# Conexión con Hugging Face (token con permiso de escritura)
from huggingface_hub import login, HfApi
login()
api = HfApi()
REPO = 'Complexhibit-Project/complexhibit-nivel1'

In [ ]:
%%writefile complexhibit_ie.py
"""Complexhibit: extracción de información de catálogos en tres niveles.

Nivel 1  bloques:   catalogueItem (ITEM) y exhibition (EXHIB)        -> modelo BIO
Nivel 2  entidades: persName, title, institution, placeName, date...  -> modelo BIO
Nivel 3  detalle:   dimensiones y fechas                              -> reglas
                    roles y tipos (person.role, institution.type...)  -> clasificador

Este módulo contiene todo lo que no depende del modelo: lectura de GATE,
conversión a BIO, reglas, ejemplos para el clasificador de atributos,
decodificación de predicciones y montaje de la salida anidada.
"""
import json
import re
import xml.etree.ElementTree as ET
from fractions import Fraction
from pathlib import Path

# ----------------------------------------------------------------------------
# Esquema (según el Manual de etiquetado v1)
# ----------------------------------------------------------------------------
# El orden de cada diccionario fija la prioridad si dos anotaciones se solapan.
NIVEL2 = {
    "persName": "PER",
    "title": "TITLE",
    "institution": "ORG",
    "venue": "ORG",
    "placeName": "LOC",
    "date": "DATE",
    "technique": "TECH",
    "material": "TECH",
    "dimensions": "DIM",
    "address": "ADDR",
    "idno": "IDNO",
}
NIVEL1 = {
    "catalogueItem": "ITEM",
    "exhibition": "EXHIB",
}
# Atributos que aprende el clasificador del nivel 3: (tipo GATE, rasgo)
ATRIBUTOS = [
    ("person", "role"),
    ("institution", "type"),
    ("institution", "role"),
    ("work", "type"),
    ("technique", "type"),
]

# Reemplazo global erróneo detectado en el corpus: al renombrar valores de
# atributos (city -> poblation, material -> technique) se modificó también
# el texto. Se corrige solo el texto de cada palabra; las posiciones no cambian.
TOKEN_FIXES = [
    (re.compile(r"poblation"), "city"),
    (re.compile(r"techniqueity"), "materiality"),
    (re.compile(r"techniquely"), "materially"),
    (re.compile(r"detechniqueiz"), "dematerializ"),
]

TOKEN_RE = re.compile(r"\w+(?:[-'’]\w+)*|[^\w\s]", re.UNICODE)


# ----------------------------------------------------------------------------
# Lectura de GATE
# ----------------------------------------------------------------------------
def load_gate(path):
    """Devuelve (texto, anotaciones) con anotaciones = [(tipo, ini, fin, rasgos)]."""
    root = ET.parse(path).getroot()
    tw = root.find("TextWithNodes")
    text = tw.text or ""
    offsets = {}
    for node in tw:
        offsets[int(node.get("id"))] = len(text)
        text += node.tail or ""
    anns = []
    for aset in root.findall("AnnotationSet"):
        for a in aset.findall("Annotation"):
            s = offsets.get(int(a.get("StartNode")), int(a.get("StartNode")))
            e = offsets.get(int(a.get("EndNode")), int(a.get("EndNode")))
            feats = {f.find("Name").text: (f.find("Value").text or "") for f in a.findall("Feature")}
            if e > s:
                anns.append((a.get("Type"), s, e, feats))
    return text, anns


def fix_token(tok):
    for pat, rep in TOKEN_FIXES:
        tok = pat.sub(rep, tok)
    return tok


def tokenize(text):
    return [(m.group(), m.start(), m.end()) for m in TOKEN_RE.finditer(text)]


# ----------------------------------------------------------------------------
# Conversión a BIO
# ----------------------------------------------------------------------------
def resolve_overlaps(spans, prioridad):
    spans = sorted(spans, key=lambda x: (prioridad.index(x[2]), -(x[1] - x[0])))
    kept = []
    for s, e, t in spans:
        if all(e <= ks or s >= ke for ks, ke, _ in kept):
            kept.append((s, e, t))
    return sorted(kept)


def to_bio(tokens, spans, mapa):
    tags = ["O"] * len(tokens)
    for s, e, t in spans:
        inside = [i for i, (_, ts, te) in enumerate(tokens) if ts < e and te > s]
        for k, i in enumerate(inside):
            tags[i] = ("B-" if k == 0 else "I-") + mapa[t]
    return tags


def segment_bounds(tokens, text, tags=None, max_tokens=200):
    """Índices [ini, fin) de fragmentos: se corta en líneas en blanco o al llegar
    a max_tokens, sin partir nunca una entidad."""
    bounds, start = [], 0
    for i, (_, s, e) in enumerate(tokens):
        nxt = tokens[i + 1] if i + 1 < len(tokens) else None
        if nxt is None:
            bounds.append((start, i + 1))
            break
        inside_next = tags is not None and tags[i + 1].startswith("I-")
        gap = text[e:nxt[1]]
        if not inside_next and ("\n\n" in gap or i + 1 - start >= max_tokens):
            bounds.append((start, i + 1))
            start = i + 1
    return bounds


def documento_bio(path, mapa, max_tokens=200):
    text, anns = load_gate(path)
    spans = [(s, e, t) for t, s, e, _ in anns if t in mapa]
    tokens = tokenize(text)
    tags = to_bio(tokens, resolve_overlaps(spans, list(mapa)), mapa)
    filas = []
    for k, (a, b) in enumerate(segment_bounds(tokens, text, tags, max_tokens)):
        filas.append({
            "doc": Path(path).stem, "seg": k,
            "tokens": [fix_token(t[0]) for t in tokens[a:b]],
            "tags": tags[a:b],
        })
    return filas


def construir_bio(carpeta, mapa, salida, max_tokens=200):
    n, stats = 0, {}
    with open(salida, "w", encoding="utf-8") as fh:
        for p in sorted(Path(carpeta).rglob("*.xml")):
            try:
                filas = documento_bio(p, mapa, max_tokens)
            except ET.ParseError as err:
                print(f"AVISO: no se pudo leer {p.name}: {err}")
                continue
            for f in filas:
                fh.write(json.dumps(f, ensure_ascii=False) + "\n")
                for t in f["tags"]:
                    if t.startswith("B-"):
                        stats[t[2:]] = stats.get(t[2:], 0) + 1
            n += 1
    print(f"{n} documentos -> {salida}")
    print("Entidades por etiqueta:", dict(sorted(stats.items(), key=lambda x: -x[1])))
    return stats


# ----------------------------------------------------------------------------
# Nivel 3a: reglas de dimensiones
# ----------------------------------------------------------------------------
_NUM = r"\d+(?:[.,]\d+)?(?:\s+\d+/\d+)?|\d+/\d+"


def _valor(s):
    s = s.strip().replace(",", ".")
    try:
        return float(sum(Fraction(p) for p in s.split()))
    except (ValueError, ZeroDivisionError):
        return None


def parse_dimensions(texto, unidad_defecto=None):
    """'93 x 19 x 16' -> {'height': 93.0, 'width': 19.0, 'depth': 16.0, 'unit': ...}
    Convención de los catálogos: alto x ancho x fondo."""
    t = texto.lower()
    out = {}
    if re.search(r"\b(lbs?|pounds?)\b", t):
        out["unit"] = "libra"
    elif re.search(r"(?<![a-záéíóú])kg\b|kilo", t):
        out["unit"] = "kg"
    elif re.search(r"\bfeet\b|(?<![a-záéíóú])ft\b|\bpies\b", t):
        out["unit"] = "feet"
    elif re.search(r"(?<![a-záéíóú])cm\b", t):
        out["unit"] = "cm"
    elif re.search(r"(?<![a-záéíóú])mm\b", t):
        out["unit"] = "mm"
    elif re.search(r"(?<![a-záéíóú])m\b|metros", t):
        out["unit"] = "m"
    elif '"' in t or "inch" in t or "pulgada" in t or "in." in t:
        out["unit"] = "inches"
    if "unit" not in out and unidad_defecto:
        out["unit"] = unidad_defecto
    nums = re.findall(_NUM, t)
    if out.get("unit") in ("libra", "kg"):
        if nums:
            out["weight"] = _valor(nums[0])
        return out
    if re.search(r"\bdi[aá]m|ø|⌀", t) and nums:
        out["diameter"] = _valor(nums[0])
        return out
    rango = re.search(rf"({_NUM})\s+(?:to|a|-)\s+({_NUM})\s+(?:feet|ft|pies|cm|m)?\s*(?:in height|de alto|alto)", t)
    if rango:
        out["height"] = (_valor(rango.group(1)), _valor(rango.group(2)))
        return out
    if re.search(r"in height|de alto|\balto\b", t) and nums:
        out["height"] = _valor(nums[0])
        return out
    partes = re.split(r"\s*[x×]\s*", t)
    vals = [_valor(m.group()) for p in partes for m in [re.search(_NUM, p)] if m]
    for clave, v in zip(("height", "width", "depth"), vals):
        out[clave] = v
    return out


def unidad_documento(text):
    """Unidad declarada para todo el catálogo, p. ej. 'All dimensions are given
    in inches' o 'Medidas en centímetros'."""
    t = text.lower()
    m = re.search(r"\b(dimensions?|measurements?|medidas|dimensiones)\b[^.\n;]{0,30}?\b(?:in|en)\s+(inches|inch|cent[ií]met\w*|cm|metros|feet)\b", t)
    if not m:
        return None
    u = m.group(2)
    return "inches" if u.startswith("inch") else "feet" if u == "feet" else "m" if u == "metros" else "cm"


def unidad_mayoritaria(frases):
    """Unidad explícita más frecuente entre las dimensiones de un catálogo: se usa
    para las medidas que no indican unidad (p. ej. '19 5/8 x 29 13/16')."""
    cuenta = {}
    for f in frases:
        u = parse_dimensions(f).get("unit")
        if u and u not in ("libra", "kg"):   # solo unidades de longitud
            cuenta[u] = cuenta.get(u, 0) + 1
    return max(cuenta, key=cuenta.get) if cuenta else None


def unidad_por_defecto(text, frases):
    """Primero la unidad declarada en el catálogo ('dimensions are given in inches');
    si no la hay, la más frecuente entre sus medidas."""
    return unidad_documento(text) or unidad_mayoritaria(frases)


def dimensiones_oro(text, anns):
    """Pares (texto de la dimensión, valores anotados) para evaluar las reglas."""
    res = []
    for t, s, e, f in anns:
        if t != "dimensions":
            continue
        oro = {"unit": f.get("unit")} if f.get("unit") else {}
        for t2, s2, e2, _ in anns:
            if t2 in ("height", "width", "depth", "weight", "diameter") and s <= s2 and e2 <= e:
                v = text[s2:e2]
                m = re.match(rf"\s*({_NUM})\s+(?:to|a|-)\s+({_NUM})\s*$", v)
                oro[t2] = (_valor(m.group(1)), _valor(m.group(2))) if m else _valor(v)
        res.append((text[s:e], oro))
    return res


# ----------------------------------------------------------------------------
# Nivel 3b: reglas de fechas (formato ISO del manual: when / from-to)
# ----------------------------------------------------------------------------
MESES = {
    "jan": 1, "january": 1, "enero": 1, "ene": 1,
    "feb": 2, "february": 2, "febrero": 2,
    "mar": 3, "march": 3, "marzo": 3,
    "apr": 4, "april": 4, "abril": 4, "abr": 4,
    "may": 5, "mayo": 5,
    "jun": 6, "june": 6, "junio": 6,
    "jul": 7, "july": 7, "julio": 7,
    "aug": 8, "august": 8, "agosto": 8, "ago": 8,
    "sep": 9, "sept": 9, "september": 9, "septiembre": 9, "setiembre": 9,
    "oct": 10, "october": 10, "octubre": 10,
    "nov": 11, "november": 11, "noviembre": 11,
    "dec": 12, "december": 12, "diciembre": 12, "dic": 12,
}
_DIAS_MES = [31, 29, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]


def _ultimo_dia(y, m):
    if m == 2:
        return 29 if (y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)) else 28
    return _DIAS_MES[m - 1]


_RANGO_INCIERTO = re.compile(r"^\W*(\d{4})\s*[-–—]\s*(\d{2}|\d{4})\s*/\s*(\d{4})\W*$")
_ANIO_ABREV = re.compile(r"\b(\d{2})(\d{2})\s*[-–—]\s*(\d{2})(?![\d/])")


def parse_date(texto):
    t = texto.lower().replace("’", "'")
    # '1920-21/1987': fecha de creación incierta (1920 o 1921) y fecha posterior (p. ej., estampación)
    m = _RANGO_INCIERTO.match(t)
    if m:
        a, b, c = m.groups()
        b = a[:2] + b if len(b) == 2 else b
        return {"notBefore": a, "notAfter": b, "to": c}
    # '1969-71' -> '1969 - 1971'
    t = _ANIO_ABREV.sub(lambda m: f"{m.group(1)}{m.group(2)} - {m.group(1)}{m.group(3)}", t)
    toks = re.findall(r"[a-záéíóúñ]+|\d+", t)
    fechas = []  # lista de [dia, mes, año] en orden de aparición
    cur = [None, None, None]
    for tok in toks:
        if tok.isdigit():
            n = int(tok)
            if len(tok) == 4 and 1000 <= n <= 2100:
                cur[2] = n
                fechas.append(cur)
                cur = [None, None, None]
            elif n <= 31:
                if cur[0] is not None:  # nuevo día sin año: cerrar el anterior
                    fechas.append(cur)
                    cur = [None, None, None]
                cur[0] = n
        elif tok in MESES:
            if cur[1] is not None:
                fechas.append(cur)
                cur = [None, None, None]
            cur[1] = MESES[tok]
    if cur != [None, None, None]:
        fechas.append(cur)
    # formatos numéricos dd/mm/aaaa
    num = [x for x in re.findall(r"(\d{1,2})[/.-](\d{1,2})[/.-](\d{4})", t) if 1 <= int(x[1]) <= 12]
    if num:
        fechas = [[int(d), int(m), int(y)] for d, m, y in num]
    fechas = [f for f in fechas if any(x is not None for x in f)]
    if not fechas:
        return {}
    # el año (y el mes) se propagan hacia atrás: "9 January-6 March, 1998"
    for i in range(len(fechas) - 2, -1, -1):
        if fechas[i][2] is None:
            fechas[i][2] = fechas[i + 1][2]
            if fechas[i][1] is None:
                fechas[i][1] = fechas[i + 1][1]
    if any(f[2] is None for f in fechas):
        return {}
    # el mes se propaga hacia delante en rangos de días: 'June 20-26, 1963'
    for i in range(1, len(fechas)):
        if fechas[i][1] is None and fechas[i][0] is not None and fechas[i - 1][1] is not None:
            fechas[i][1] = fechas[i - 1][1]

    def iso(f, fin=False):
        d, m, y = f
        if m is None:
            return f"{y:04d}" if not fin else f"{y:04d}"
        if d is None:
            d = _ultimo_dia(y, m) if fin else 1
        return f"{y:04d}-{m:02d}-{d:02d}"

    if len(fechas) == 1:
        f = fechas[0]
        if f[1] is None:
            return {"when": f"{f[2]:04d}"}
        if f[0] is None:  # mes y año: intervalo del mes
            return {"from": iso(f), "to": iso(f, fin=True)}
        return {"when": iso(f)}
    return {"from": iso(fechas[0]), "to": iso(fechas[-1], fin=True)}


def fechas_oro(text, anns):
    res = []
    for t, s, e, f in anns:
        if t == "date":
            oro = {k: v for k, v in f.items() if k in ("when", "from", "to", "notBefore", "notAfter")}
            if oro:
                res.append((text[s:e], oro))
    return res


def evaluar_reglas(carpeta, docs=None):
    """Exactitud de las reglas frente a los valores anotados a mano."""
    dim_ok = dim_n = fec_ok = fec_n = 0
    errores = {"dimensiones": [], "fechas": []}
    for p in sorted(Path(carpeta).rglob("*.xml")):
        if docs is not None and p.stem not in docs:
            continue
        try:
            text, anns = load_gate(p)
        except ET.ParseError:
            continue
        pares = dimensiones_oro(text, anns)
        udoc = unidad_por_defecto(text, [f for f, _ in pares])
        for frase, oro in pares:
            pred = parse_dimensions(frase, udoc)
            campos = [k for k in oro if k != "unit"]
            ok = all(pred.get(k) == oro[k] for k in campos) and (not oro.get("unit") or pred.get("unit") == oro["unit"])
            dim_n += 1
            dim_ok += ok
            if not ok and len(errores["dimensiones"]) < 30:
                errores["dimensiones"].append({"texto": frase, "oro": oro, "regla": pred, "doc": p.stem})
        for frase, oro in fechas_oro(text, anns):
            pred = parse_date(frase)
            ok = pred == oro
            fec_n += 1
            fec_ok += ok
            if not ok and len(errores["fechas"]) < 30:
                errores["fechas"].append({"texto": frase, "oro": oro, "regla": pred, "doc": p.stem})
    return {
        "dimensiones": {"n": dim_n, "exactitud": dim_ok / dim_n if dim_n else None},
        "fechas": {"n": fec_n, "exactitud": fec_ok / fec_n if fec_n else None},
        "errores": errores,
    }


# ----------------------------------------------------------------------------
# Nivel 3c: ejemplos para el clasificador de roles y tipos
# ----------------------------------------------------------------------------
VENTANA = 250  # caracteres de contexto a cada lado


def con_marcas(text, s, e, ventana=VENTANA):
    a, b = max(0, s - ventana), min(len(text), e + ventana)
    frag = text[a:s] + " [E] " + text[s:e] + " [/E] " + text[e:b]
    frag = " ".join(fix_token(w) for w in frag.split())
    return frag


def ejemplos_atributos(carpeta):
    filas = []
    for p in sorted(Path(carpeta).rglob("*.xml")):
        try:
            text, anns = load_gate(p)
        except ET.ParseError:
            continue
        for t, s, e, f in anns:
            for tipo, rasgo in ATRIBUTOS:
                if t == tipo and f.get(rasgo, "").strip():
                    valor = re.sub(r"\s*,\s*", ", ", f[rasgo].strip().lstrip("_"))
                    filas.append({
                        "doc": p.stem, "atributo": f"{tipo}.{rasgo}",
                        "texto": f"{tipo}.{rasgo}: " + con_marcas(text, s, e),
                        "valor": valor,
                    })
    return filas


# ----------------------------------------------------------------------------
# Inferencia: de etiquetas por palabra a fragmentos y a una salida anidada
# ----------------------------------------------------------------------------
def bio_a_spans(tokens, tags):
    """tokens = [(palabra, ini, fin)], tags = ['B-PER', ...] -> [(ini, fin, 'PER')]"""
    spans, cur = [], None
    for (w, s, e), tag in zip(tokens, tags):
        if tag.startswith("B-") or (tag.startswith("I-") and (cur is None or cur[2] != tag[2:])):
            if cur:
                spans.append(tuple(cur))
            cur = [s, e, tag[2:]]
        elif tag.startswith("I-") and cur:
            cur[1] = e
        else:
            if cur:
                spans.append(tuple(cur))
            cur = None
    if cur:
        spans.append(tuple(cur))
    return spans


def montar_salida(text, spans_n1, spans_n2, atributos=None):
    """Combina niveles 1, 2 y 3 en una estructura anidada (al estilo del manual).
    atributos: dict {(ini, fin): {'person.role': 'creator', ...}}"""
    atributos = atributos or {}
    udoc = unidad_por_defecto(text, [text[s:e] for s, e, lab in spans_n2 if lab == "DIM"])

    def entidad(s, e, lab):
        d = {"tipo": lab, "texto": text[s:e].strip(), "ini": s, "fin": e}
        if lab == "DIM":
            d["valores"] = parse_dimensions(text[s:e], udoc)
        if lab == "DATE":
            d["valores"] = parse_date(text[s:e])
        if (s, e) in atributos:
            d["atributos"] = atributos[(s, e)]
        return d

    bloques = []
    for s, e, lab in spans_n1:
        hijos = [entidad(s2, e2, l2) for s2, e2, l2 in spans_n2 if s <= s2 and e2 <= e]
        b = {"tipo": {"ITEM": "catalogueItem", "EXHIB": "exhibition"}.get(lab, lab),
             "texto": text[s:e].strip()[:300], "ini": s, "fin": e, "contenido": hijos}
        if (s, e) in atributos:
            b["atributos"] = atributos[(s, e)]
        bloques.append(b)
    dentro = {(s2, e2) for s, e, _ in spans_n1 for s2, e2, _ in spans_n2 if s <= s2 and e2 <= e}
    sueltas = [entidad(s, e, l) for s, e, l in spans_n2 if (s, e) not in dentro]
    return {"bloques": bloques, "entidades_fuera_de_bloques": sueltas}


In [ ]:
def segment_bounds_inferencia(tokens, text, max_tokens=200, min_tokens=120):
    """Fragmentos para la inferencia, sin conocer las entidades: no se corta en
    cada línea en blanco (eso partiría las fichas que llevan una línea en blanco
    entre la imagen y el pie), sino solo al acercarse a max_tokens, eligiendo el
    salto de línea más «fuerte» (preferiblemente doble) a partir de min_tokens."""
    bounds, start, n = [], 0, len(tokens)
    while start < n:
        if n - start <= max_tokens:
            bounds.append((start, n)); break
        mejor, puntos = None, -1
        for i in range(start + min_tokens, min(start + max_tokens, n - 1)):
            gap = text[tokens[i][2]:tokens[i + 1][1]]
            p = gap.count("\n")
            if p > 0 and p >= puntos:
                mejor, puntos = i + 1, p
        corte = mejor or start + max_tokens
        bounds.append((start, corte)); start = corte
    return bounds

# Entrada: ZIP con los XML de GATE
import zipfile, os, glob, json, unicodedata, hashlib
from google.colab import files
print('Selecciona el ZIP con los XML de GATE')
sub = files.upload()
os.makedirs('xml', exist_ok=True)
for n in sub:
    zipfile.ZipFile(n).extractall('xml')
XMLS = sorted(p for p in glob.glob('xml/**/*.xml', recursive=True) if not os.path.basename(p).startswith('._'))
nfc = lambda s: unicodedata.normalize('NFC', s)
# Misma partición que la versión 1.0 (semilla 42)
PARTICION = {"test": ["1. ) TRANSFIGURACIONES( fabio gomes", "16. WORDS AND IMAGES", "3. April Gornik", "Catalogo-Gonzalo-Chillida", "Catalogo-Joan-Bueno-galeria-cortabitarte", "Catálogo-Carmen-Nieto", "c2b76d7a0dfdf25df06b7d0baba7ca4b659cebd5", "catalogo-agustinhernandez", "catalogo-penalver-web", "extracto-catalogo-morandi-vol-ii"], "validation": ["180222-CATALOGO-EXPOSICION-RAFAEL-VIDAL", "Catalogo virtual web", "CatalogoExposicionJavierBuzon", "Catalogo_de_la_exposicion_de_Fernando_Lo", "Guillermo-Simon---Genesis---Catalogo-exposicion-en-Malvin-Gallery", "LIMEN-Catalogo-Art-Madrid", "catalogo-de-madera-y-color-web-1", "catalogo-el-abrazo-1-_compressed", "catalogo-exposicion-fatima-gomez-aurea-convento-capuchinos", "fernando_palacios___catalogo_open_series__pintura.."]}
TEST = {nfc(d) for d in PARTICION['test']}; VAL = {nfc(d) for d in PARTICION['validation']}
particion = lambda stem: 'test' if nfc(stem) in TEST else ('validation' if nfc(stem) in VAL else 'train')
print(len(XMLS), 'ficheros XML')

In [ ]:
# Datos: fragmentos cortados SIN mirar la anotación
import complexhibit_ie as ie, collections
DOCS = []
for p in XMLS:
    try:
        texto, anns = ie.load_gate(p)
    except Exception as err:
        print('Omitido', os.path.basename(p), err); continue
    stem = os.path.splitext(os.path.basename(p))[0]
    spans = [(s, e, t) for t, s, e, _ in anns if t in ie.NIVEL1]
    if not anns: continue
    tokens = ie.tokenize(texto)
    tags = ie.to_bio(tokens, ie.resolve_overlaps(spans, list(ie.NIVEL1)), ie.NIVEL1)
    DOCS.append({'stem': stem, 'split': particion(stem), 'texto': texto, 'tokens': tokens, 'tags': tags})
FILAS = []
for d in DOCS:
    for a, b in segment_bounds_inferencia(d['tokens'], d['texto']):
        FILAS.append({'split': d['split'], 'tokens': [ie.fix_token(t[0]) for t in d['tokens'][a:b]], 'tags': d['tags'][a:b]})
print(collections.Counter(d['split'] for d in DOCS), collections.Counter(f['split'] for f in FILAS))
assert sum(d['split'] == 'test' for d in DOCS) == 10, 'No se han encontrado los 10 catálogos de prueba'

In [ ]:
# Evaluación en condiciones de uso real: catálogo completo, sin mirar la anotación
import numpy as np
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

def predecir(tok, modelo, d, max_tokens=200):
    tokens = d['tokens']; tags = ['O'] * len(tokens); id2 = modelo.config.id2label
    for a, b in segment_bounds_inferencia(tokens, d['texto'], max_tokens):
        enc = tok([ie.fix_token(t[0]) for t in tokens[a:b]], is_split_into_words=True, truncation=True,
                  max_length=512, return_tensors='pt').to(modelo.device)
        with torch.no_grad():
            pred = modelo(**enc).logits.argmax(-1)[0].tolist()
        prev = None
        for j, w in enumerate(enc.word_ids(0)):
            if w is not None and w != prev: tags[a + w] = id2[pred[j]]
            prev = w
    return tags

def iou(x, y):
    return max(0, min(x[1], y[1]) - max(x[0], y[0])) / (max(x[1], y[1]) - min(x[0], y[0]))

def evaluar(tok, modelo, split):
    G, P, res = [], [], collections.defaultdict(lambda: collections.Counter())
    for d in [d for d in DOCS if d['split'] == split]:
        pt = predecir(tok, modelo, d); G.append(d['tags']); P.append(pt)
        gs = ie.bio_a_spans(d['tokens'], d['tags']); ps = ie.bio_a_spans(d['tokens'], pt)
        for lab in ('ITEM', 'EXHIB'):
            g = [x for x in gs if x[2] == lab]; p = [x for x in ps if x[2] == lab]
            r = res[lab]; r['oro'] += len(g); r['pred'] += len(p)
            r['exacta'] += len({(x[0], x[1]) for x in g} & {(x[0], x[1]) for x in p})
            r['solape_oro'] += sum(any(iou(x, y) >= .5 for y in p) for x in g)
            r['solape_pred'] += sum(any(iou(x, y) >= .5 for x in g) for y in p)
    out = {}
    for lab, r in list(res.items()) + [('total', sum(res.values(), collections.Counter()))]:
        pe = r['exacta'] / r['pred'] if r['pred'] else 0; re_ = r['exacta'] / r['oro'] if r['oro'] else 0
        ps_ = r['solape_pred'] / r['pred'] if r['pred'] else 0; rs = r['solape_oro'] / r['oro'] if r['oro'] else 0
        f = lambda a, b: 2 * a * b / (a + b) if a + b else 0
        out[lab] = {'n_oro': r['oro'], 'n_pred': r['pred'],
                    'exacta': {'precision': round(pe, 3), 'recall': round(re_, 3), 'f1': round(f(pe, re_), 3)},
                    'solapamiento_50': {'precision': round(ps_, 3), 'recall': round(rs, 3), 'f1': round(f(ps_, rs), 3)}}
    return out

def tabla(nombre, r):
    print(f'--- {nombre} ---')
    for lab, v in r.items():
        print(f"{lab:6} oro={v['n_oro']:4} pred={v['n_pred']:4} | exacta P {v['exacta']['precision']:.2f} R {v['exacta']['recall']:.2f} F1 {v['exacta']['f1']:.2f}"
              f" | solapamiento P {v['solapamiento_50']['precision']:.2f} R {v['solapamiento_50']['recall']:.2f} F1 {v['solapamiento_50']['f1']:.2f}")

In [ ]:
# 1) Referencia: modelo publicado (v1.0) evaluado en condiciones de uso real
from transformers import AutoTokenizer, AutoModelForTokenClassification
tok_v10 = AutoTokenizer.from_pretrained(REPO)
m_v10 = AutoModelForTokenClassification.from_pretrained(REPO).to('cuda' if torch.cuda.is_available() else 'cpu').eval()
RES = {'v1.0': {'test': evaluar(tok_v10, m_v10, 'test'), 'validation': evaluar(tok_v10, m_v10, 'validation')}}
tabla('v1.0 · prueba', RES['v1.0']['test']); tabla('v1.0 · validación', RES['v1.0']['validation'])
del m_v10; import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 2) Entrenamiento de la versión 1.1 (unos 30-45 minutos en L4)
import inspect
from datasets import Dataset, DatasetDict
from transformers import TrainingArguments as _TA, Trainer, DataCollatorForTokenClassification, set_seed
_adm = set(inspect.signature(_TA.__init__).parameters)
TrainingArguments = lambda **kw: _TA(**{k: v for k, v in kw.items() if k in _adm})
set_seed(42)
BASE = 'FacebookAI/xlm-roberta-large'
tok = AutoTokenizer.from_pretrained(BASE)
ETQ = ['O', 'B-EXHIB', 'I-EXHIB', 'B-ITEM', 'I-ITEM']; L2I = {e: i for i, e in enumerate(ETQ)}
ds = DatasetDict({s: Dataset.from_list([{'tokens': f['tokens'], 'labels': [L2I[t] for t in f['tags']]} for f in FILAS if f['split'] == s])
                  for s in ('train', 'validation')})
def alinear(ej):
    enc = tok(ej['tokens'], is_split_into_words=True, truncation=True, max_length=512); todas = []
    for i, labs in enumerate(ej['labels']):
        prev, ids = None, []
        for w in enc.word_ids(batch_index=i):
            ids.append(-100 if w is None or w == prev else labs[w]); prev = w
        todas.append(ids)
    enc['labels'] = todas
    return enc
tds = ds.map(alinear, batched=True, remove_columns=['tokens', 'labels'])
def metricas(ev):
    pr = np.argmax(ev[0], -1); P, L = [], []
    for p, l in zip(pr, ev[1]):
        P.append([ETQ[a] for a, b in zip(p, l) if b != -100]); L.append([ETQ[b] for a, b in zip(p, l) if b != -100])
    return {'precision': precision_score(L, P), 'recall': recall_score(L, P), 'f1': f1_score(L, P)}
modelo = AutoModelForTokenClassification.from_pretrained(BASE, num_labels=len(ETQ), id2label=dict(enumerate(ETQ)), label2id=L2I)
args = TrainingArguments(output_dir='tmp_n1', learning_rate=2e-5, num_train_epochs=6, per_device_train_batch_size=4,
                         gradient_accumulation_steps=2, per_device_eval_batch_size=8, warmup_steps=300, weight_decay=0.01,
                         eval_strategy='epoch', evaluation_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
                         metric_for_best_model='f1', save_total_limit=1, fp16=torch.cuda.is_available(), report_to='none', seed=42)
tr = Trainer(model=modelo, args=args, train_dataset=tds['train'], eval_dataset=tds['validation'],
             data_collator=DataCollatorForTokenClassification(tok), compute_metrics=metricas)
tr.train()
modelo = tr.model.eval()

In [ ]:
# 3) Evaluación de la versión 1.1 en las mismas condiciones
RES['v1.1'] = {'test': evaluar(tok, modelo, 'test'), 'validation': evaluar(tok, modelo, 'validation')}
tabla('v1.1 · prueba', RES['v1.1']['test']); tabla('v1.1 · validación', RES['v1.1']['validation'])
f_old = RES['v1.0']['test']['total']['solapamiento_50']['f1'] + RES['v1.0']['test']['total']['exacta']['f1']
f_new = RES['v1.1']['test']['total']['solapamiento_50']['f1'] + RES['v1.1']['test']['total']['exacta']['f1']
MEJORA = f_new > f_old
print('\nLa versión 1.1', 'MEJORA' if MEJORA else 'NO mejora', 'a la 1.0 en los catálogos de prueba')
json.dump(RES, open('metricas_nivel1_v11.json', 'w'), indent=1)

In [ ]:
# 4) Publicación en Hugging Face como nueva versión (solo si mejora)
import shutil
if MEJORA:
    dest = '/content/nivel1_v11'; shutil.rmtree(dest, ignore_errors=True)
    modelo.half().save_pretrained(dest); tok.save_pretrained(dest); modelo.float()
    json.dump(RES['v1.1'], open(f'{dest}/metricas.json', 'w'), indent=1)
    api.upload_folder(folder_path=dest, repo_id=REPO, commit_message='v1.1: segmentación del texto sin usar la anotación, igual en entrenamiento, evaluación y uso real')
    print('Publicada la versión 1.1 en https://huggingface.co/' + REPO)
else:
    print('No se publica: se mantiene la versión 1.0')

In [ ]:
# 5) Predicciones del nivel 1 en los 100 catálogos y resultados
os.makedirs('pred_n1', exist_ok=True)
for d in DOCS:
    pt = predecir(tok, modelo, d)
    json.dump({'nombre_original': d['stem'], 'particion': d['split'], 'version': 'v1.1',
               'sha256_texto': hashlib.sha256(d['texto'].encode('utf-8')).hexdigest(),
               'predicciones': [{'nivel': 1, 'tipo': l, 'ini': s, 'fin': e} for s, e, l in ie.bio_a_spans(d['tokens'], pt)]},
              open(f"pred_n1/{hashlib.sha256(d['texto'].encode('utf-8')).hexdigest()[:12]}.json", 'w'), ensure_ascii=False, indent=1)
shutil.copy('metricas_nivel1_v11.json', 'pred_n1/')
shutil.make_archive('resultados_nivel1_v11', 'zip', 'pred_n1')
print('Resultados: /content/resultados_nivel1_v11.zip — descárgalo desde el panel de archivos (⋮ → Descargar)')